# Colab Demo: Baseline RAG with Long-T5

This notebook runs the baseline RAG pipeline with a fixed generator model `google/long-t5-tglobal-base` using prebuilt artifacts from Google Drive.

Outputs:
- On-screen qualitative results (query, evidence, response, claims)
- JSON file under `outputs/rag_demo_results_<timestamp>.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/AIST-FYP')
DRIVE_REPO_MIRROR = Path('/content/drive/MyDrive/AIST-FYP')
REPO_URL = 'https://github.com/<your-org-or-user>/AIST-FYP.git'  # update if cloning from GitHub

if REPO_DIR.exists():
    print(f'Repository already exists at {REPO_DIR}')
elif DRIVE_REPO_MIRROR.exists():
    print('Copying repository snapshot from Google Drive...')
    os.system(f'cp -r {DRIVE_REPO_MIRROR} {REPO_DIR}')
else:
    print('Cloning repository from GitHub...')
    clone_code = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if clone_code != 0:
        raise RuntimeError('Failed to prepare /content/AIST-FYP. Update REPO_URL or place a repo snapshot in Drive.')

%cd /content/AIST-FYP

In [ ]:
import os

print('Installing dependencies...')
uv_code = os.system('pip -q install uv')
sync_code = os.system('uv sync --project colab/env --extra evaluation') if uv_code == 0 else 1

if sync_code != 0:
    print('uv sync failed, using pip fallback')
    os.system('python -m pip install -q --upgrade pip')
    os.system('python -m pip install -q -r requirements.txt')

venv_bin = '/content/AIST-FYP/colab/env/.venv/bin'
if os.path.exists(venv_bin):
    os.environ['PATH'] = venv_bin + ':' + os.environ['PATH']

os.system('python -m spacy download en_core_web_sm')

In [ ]:
import torch

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA is not available. Switch Colab runtime to GPU before running the demo.')

In [ ]:
from pathlib import Path
import shutil

STRATEGY = 'development'  # one of: development, validation, production
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/AIST-FYP')
LOCAL_PROJECT_ROOT = Path('/content/AIST-FYP')

required_files = [
    DRIVE_PROJECT_ROOT / f'data/indexes/{STRATEGY}/faiss.index',
    DRIVE_PROJECT_ROOT / f'data/indexes/{STRATEGY}/metadata.pkl',
    DRIVE_PROJECT_ROOT / f'data/indexes/{STRATEGY}/bm25_index.pkl',
    DRIVE_PROJECT_ROOT / f'data/processed/wiki_chunks_{STRATEGY}.jsonl',
]

missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required Drive artifacts:\n' + '\n'.join(missing))

local_index_dir = LOCAL_PROJECT_ROOT / f'data/indexes/{STRATEGY}'
local_processed_dir = LOCAL_PROJECT_ROOT / 'data/processed'
local_index_dir.mkdir(parents=True, exist_ok=True)
local_processed_dir.mkdir(parents=True, exist_ok=True)

for file_name in ['faiss.index', 'metadata.pkl', 'bm25_index.pkl']:
    src = DRIVE_PROJECT_ROOT / f'data/indexes/{STRATEGY}/{file_name}'
    dst = local_index_dir / file_name
    shutil.copy2(src, dst)

processed_src = DRIVE_PROJECT_ROOT / f'data/processed/wiki_chunks_{STRATEGY}.jsonl'
processed_dst = LOCAL_PROJECT_ROOT / f'data/processed/wiki_chunks_{STRATEGY}.jsonl'
shutil.copy2(processed_src, processed_dst)

print('Artifact sync complete for strategy:', STRATEGY)

In [ ]:
import yaml
from pathlib import Path

BASE_CONFIG = Path('/content/AIST-FYP/config.yaml')
COLAB_CONFIG = Path('/content/AIST-FYP/config.colab.longt5.yaml')

with open(BASE_CONFIG, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['models']['generator'] = 'google/long-t5-tglobal-base'
cfg['processing']['device'] = 'cuda'
cfg['generation']['load_in_8bit'] = False

# Safety defaults for Colab stability (adjust as needed)
cfg['generation']['max_new_tokens'] = 256
cfg['retrieval']['top_k'] = 5

with open(COLAB_CONFIG, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Wrote runtime config:', COLAB_CONFIG)

In [ ]:
import sys
import json
import time
from datetime import datetime
from pathlib import Path
import numpy as np

sys.path.insert(0, '/content/AIST-FYP')

from src.pipelines import BaselineRAGPipeline

def make_json_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_serializable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj

pipeline = BaselineRAGPipeline.from_config(
    config_path='/content/AIST-FYP/config.colab.longt5.yaml',
    strategy=STRATEGY
)

sample_queries = [
    'What is artificial intelligence?',
    'How do machines learn from data?',
    'What is machine learning? How does it differ from traditional programming?'
]

all_results = []
for i, query in enumerate(sample_queries, start=1):
    t0 = time.time()
    result = pipeline.run(query, top_k=cfg['retrieval']['top_k'])
    elapsed = time.time() - t0

    all_results.append({
        'query_index': i,
        'query': query,
        'timestamp': datetime.now().isoformat(),
        'latency_sec': round(elapsed, 3),
        'result': make_json_serializable(result),
    })

print(f'Completed {len(all_results)} queries')

In [ ]:
for item in all_results:
    print('=' * 100)
    print(f"Query {item['query_index']}: {item['query']}")
    print('-' * 100)

    result = item['result']
    retrieval = result.get('retrieval_metadata', {})
    pairs = result.get('claim_evidence_pairs', [])

    print('Latency (sec):', item.get('latency_sec'))
    print('Retrieved chunks:', retrieval.get('num_retrieved'))
    print('Top score:', retrieval.get('top_score'))

    print('Draft response:')
    print(result.get('draft_response', ''))

    print('Claims extracted:', len(pairs))
    if pairs:
        first_pair = pairs[0]
        spans = first_pair.get('evidence_spans', [])
        print('Top evidence candidates (up to 2):')
        for span in spans[:2]:
            text = span.get('text', '')
            print(f"- {span.get('doc_id')}#{span.get('sent_id')}: {text[:180]}")

    print()

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import shutil

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
local_outputs = Path('/content/AIST-FYP/outputs')
local_outputs.mkdir(parents=True, exist_ok=True)
output_file = local_outputs / f'rag_demo_results_{ts}.json'

# Keep the same top-level shape as scripts/demo_baseline_rag.py
json_results = []
for item in all_results:
    json_results.append({
        'query_index': item['query_index'],
        'query': item['query'],
        'timestamp': item['timestamp'],
        'result': item['result'],
    })

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(json_results, f, ensure_ascii=False, indent=2)

print('Saved:', output_file)

# Optional: copy output back to Drive
drive_outputs = Path('/content/drive/MyDrive/AIST-FYP/outputs')
drive_outputs.mkdir(parents=True, exist_ok=True)
drive_file = drive_outputs / output_file.name
shutil.copy2(output_file, drive_file)
print('Copied to Drive:', drive_file)

## Troubleshooting

- If you hit OOM, reduce `cfg['generation']['max_new_tokens']` to 128 and run fewer queries.
- If artifacts are missing, verify `DRIVE_PROJECT_ROOT` and selected `STRATEGY`.
- If model load fails, re-check GPU runtime and rerun setup cells.